# Convert Raw Dumps to Structured HDF5 File

## Imports

In [ ]:
import os
import time
global notebook
notebook = 1

dir = '/global/u1/j/jackh/bh/harm2d'
os.chdir(dir)

import yaml
with open('config.yml', 'r') as file
    config = yaml.safe_load(file)

%run -i setup.py build_ext --inplace
%run -i pp.py build_ext --inplace
%matplotlib inline
import matplotlib
print('Imports done.')

Imports done.


## Set Relevant Settings

In [7]:
global notebook, axisym,set_cart,axisym,REF_1,REF_2,REF_3,set_cart,D,print_fieldlines
global lowres1,lowres2,lowres3, RAD_M1, RESISTIVE, export_raytracing_GRTRANS, export_raytracing_RAZIEH,r1,r2,r3
global r_min, r_max, theta_min, theta_max, phi_min,phi_max, do_griddata, do_box, check_files, kerr_schild

dir = '/pscratch/sd/l/lalakos/ml_data_rc300/reduced'
os.chdir(dir)

# set params
lowres1 = 1
lowres2 = 1
lowres3 = 1

do_box=0
r_min=1.0
r_max=100.0
theta_min=0.0
theta_max=9
phi_min=-1
phi_max=9
axisym=1
print_fieldlines=0
export_raytracing_GRTRANS=0
export_raytracing_RAZIEH=0
kerr_schild=0
DISK_THICKNESS=0.03
set_cart=0
set_mpi(0)
check_files=1
notebook=1
interpolate_var=0
AMR = 0 # get all data in grid

## Write to Disk

In [ ]:
import os
import numpy as np
import h5py
from tqdm import tqdm

# populate HDF5 file with data from source dumps
def populate_h5(
        dumps_path:str, 
        data_path:str, 
        write_batch_size: int,
        start_dump: int, 
        num_dumps: int
    ):
    with h5py.File(data_path, "a") as f:
        rblock_new_ml()

        prog_bar = tqdm(range(start_dump, start_dump+num_dumps))
        for dump in prog_bar:
            if (dump % write_batch_size) == 0:
                data_batch_array = []
                label_batch_array = []

            rpar_new(dump)
            if dump == start_dump:
                rgdump_griddata(dumps_path)
            rdump_griddata(dumps_path, dump)

            # construct array
            new_data = np.expand_dims(np.concatenate((np.log(rho), ug, np.squeeze(uu[1:4], axis=1), np.squeeze(B[1:4], axis=1)), axis=0),0)
            new_label = np.array([[dump]])

            data_batch_array.append(new_data)
            label_batch_array.append(new_label)

            if (dump % write_batch_size) == (write_batch_size - 1) or (dump > start_dump+num_dumps - write_batch_size) :
                data_batch_array = np.stack(data_batch_array, axis=0)
                label_batch_array = np.stack(label_batch_array, axis=0)
            
                if dump == start_dump:
                    f.create_dataset('data', data=data_batch_array, compression="gzip", chunks=True, maxshape=(None,8,224,48,96))
                    f.create_dataset('dump_index', data=label_batch_array, compression="gzip", chunks=True, maxshape=(None,1))
                else:
                    f['data'].resize((f['data'].shape[0] + data_batch_array.shape[0]), axis=0)
                    f['data'][-data_batch_array.shape[0]:] = data_batch_array
                
                    f['dump_index'].resize((f['dump_index'].shape[0] + label_batch_array.shape[0]), axis=0)
                    f['dump_index'][-label_batch_array.shape[0]:] = label_batch_array

                    prog_bar.set_description(f"Dump {dump} written.")


# DUMPS_PATH is the source file for the dumps to save
DUMPS_PATH = '/pscratch/sd/l/lalakos/ml_data_rc300/reduced'
# DATA_PATH is where the organized HDF5 file will be saved
DATA_PATH = os.getenv('SCRATCH')+"/data.hdf5"

num_dumps = config['end_dump'] - config['start_dump']
populate_h5(
    dups_path = DUMPS_PATH, 
    data_path = DATA_PATH, 
    write_batch_size = 32,
    start_dump = config['start_dump'], 
    num_dumps = num_dumps
)